# 11 · HPO — Faster R-CNN Swin-T
Persistent two-stage random search. Official validation is never used for tuning.

In [ ]:
DATASET_TRACK = "2class"
START_HPO = False

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_PATH = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_PATH / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", "main", "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git", str(REPO_PATH)], check=True)
    elif subprocess.check_output(["git", "-C", str(REPO_PATH), "status", "--porcelain"], text=True).strip():
        raise RuntimeError("Refusing to update a dirty Colab checkout")
    else:
        subprocess.run(["git", "-C", str(REPO_PATH), "pull", "--ff-only", "origin", "main"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_PATH / "requirements-hpo-colab.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_PATH), "--no-deps"], check=True)
else:
    REPO_PATH = Path.cwd()
sys.path.insert(0, str(REPO_PATH))
DRIVE_ROOT = Path(os.environ.get("VISDRONE_DRIVE_ROOT", "/content/drive/MyDrive/visdrone_architecture_benchmark"))

In [ ]:
MODEL_ID = "faster_rcnn_swin_t"
if SMOKE_TEST:
    result = {"status": "guarded", "model_id": MODEL_ID, "dataset_track": DATASET_TRACK}
else:
    from src.workflows.environment import ensure_model_environment
    from src.hpo.workflow import TwoStageRandomHPO
    environment = ensure_model_environment(MODEL_ID, REPO_PATH, DRIVE_ROOT)
    result = TwoStageRandomHPO(REPO_PATH, DRIVE_ROOT, MODEL_ID, DATASET_TRACK).run(start_expensive_stage=START_HPO)
result